1. Importation des bibliothèques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Pour la reproductibilité
np.random.seed(42)

2. Chargement de l'ensemble de données MNIST

In [ ]:
# Chargement des données
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Affichage des formes
print(f"Forme de x_train: {x_train.shape}")  # (60000, 28, 28)
print(f"Forme de y_train: {y_train.shape}")  # (60000,)
print(f"Forme de x_test: {x_test.shape}")    # (10000, 28, 28)
print(f"Forme de y_test: {y_test.shape}")    # (10000,)

# Visualisation de quelques exemples
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f'Label: {y_train[i]}')
    plt.axis('off')
plt.tight_layout()
plt.show()

3. Prétraitement pour le réseau entièrement connecté

In [ ]:
# Aplatissement des images
x_train_flat = x_train.reshape(x_train.shape[0], 784)
x_test_flat = x_test.reshape(x_test.shape[0], 784)

# Normalisation
x_train_flat = x_train_flat.astype('float32') / 255.0
x_test_flat = x_test_flat.astype('float32') / 255.0

# Encodage one-hot des labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"x_train_flat forme: {x_train_flat.shape}")  # (60000, 784)
print(f"y_train_cat forme: {y_train_cat.shape}")    # (60000, 10)

4. Modèle entièrement connecté (Dense Network)

In [ ]:
# Construction du modèle
model_dense = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# Compilation
model_dense.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Affichage du résumé
model_dense.summary()

# Entraînement
history_dense = model_dense.fit(
    x_train_flat, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_data=(x_test_flat, y_test_cat),
    verbose=1
)

# Évaluation
test_loss_dense, test_acc_dense = model_dense.evaluate(x_test_flat, y_test_cat, verbose=0)
print(f"Précision du modèle dense sur le test: {test_acc_dense:.4f}")

5. Prétraitement pour le CNN

In [ ]:
# Reshape pour Conv2D (ajout de la dimension canal)
x_train_cnn = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test_cnn = x_test.reshape(x_test.shape[0], 28, 28, 1)

# Normalisation
x_train_cnn = x_train_cnn.astype('float32') / 255.0
x_test_cnn = x_test_cnn.astype('float32') / 255.0

# Les labels sont déjà en one-hot (y_train_cat, y_test_cat)

print(f"x_train_cnn forme: {x_train_cnn.shape}")  # (60000, 28, 28, 1)

6. Modèle CNN (Convolutional Neural Network)

In [ ]:
# Construction du CNN
model_cnn = Sequential([
    # Première couche convolutionnelle
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    
    # Deuxième couche convolutionnelle
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Troisième couche convolutionnelle
    Conv2D(64, (3, 3), activation='relu'),
    
    # Aplatissement et couches denses
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),  # Pour éviter le surapprentissage
    Dense(10, activation='softmax')
])

# Compilation
model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Affichage du résumé
model_cnn.summary()

# Entraînement
history_cnn = model_cnn.fit(
    x_train_cnn, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_data=(x_test_cnn, y_test_cat),
    verbose=1
)

# Évaluation
test_loss_cnn, test_acc_cnn = model_cnn.evaluate(x_test_cnn, y_test_cat, verbose=0)
print(f"Précision du CNN sur le test: {test_acc_cnn:.4f}")

7. Comparaison des performances

In [ ]:
# Affichage des résultats
print("=" * 50)
print("COMPARAISON DES PERFORMANCES")
print("=" * 50)
print(f"Modèle Dense (fully connected):  {test_acc_dense:.4f}")
print(f"Modèle CNN:                      {test_acc_cnn:.4f}")
print(f"Différence:                      {test_acc_cnn - test_acc_dense:.4f}")

# Visualisation des courbes d'apprentissage
plt.figure(figsize=(12, 5))

# Précision
plt.subplot(1, 2, 1)
plt.plot(history_dense.history['accuracy'], label='Dense Train', color='blue')
plt.plot(history_dense.history['val_accuracy'], label='Dense Val', color='blue', linestyle='--')
plt.plot(history_cnn.history['accuracy'], label='CNN Train', color='red')
plt.plot(history_cnn.history['val_accuracy'], label='CNN Val', color='red', linestyle='--')
plt.title('Précision par époque')
plt.xlabel('Époque')
plt.ylabel('Précision')
plt.legend()
plt.grid(True)

# Perte
plt.subplot(1, 2, 2)
plt.plot(history_dense.history['loss'], label='Dense Train', color='blue')
plt.plot(history_dense.history['val_loss'], label='Dense Val', color='blue', linestyle='--')
plt.plot(history_cnn.history['loss'], label='CNN Train', color='red')
plt.plot(history_cnn.history['val_loss'], label='CNN Val', color='red', linestyle='--')
plt.title('Perte par époque')
plt.xlabel('Époque')
plt.ylabel('Perte')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

8. Prédictions et visualisation

In [ ]:
# Prédictions sur quelques exemples de test
n_samples = 10
indices = np.random.choice(len(x_test), n_samples, replace=False)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(indices):
    plt.subplot(2, n_samples, i+1)
    plt.imshow(x_test[idx], cmap='gray')
    plt.axis('off')
    
    # Prédictions
    pred_dense = model_dense.predict(x_test_flat[idx:idx+1], verbose=0)
    pred_cnn = model_cnn.predict(x_test_cnn[idx:idx+1], verbose=0)
    
    pred_dense_label = np.argmax(pred_dense)
    pred_cnn_label = np.argmax(pred_cnn)
    
    plt.title(f'Vrai: {y_test[idx]}\nDense: {pred_dense_label}\nCNN: {pred_cnn_label}')
    
    # Probabilités
    plt.subplot(2, n_samples, n_samples+i+1)
    plt.bar(range(10), pred_cnn[0])
    plt.xticks(range(10))
    plt.ylim(0, 1)
    plt.title(f'CNN Probas pour {y_test[idx]}')

plt.tight_layout()
plt.show()

9. Analyse des résultats

In [ ]:
# Analyse détaillée
print("\n" + "=" * 50)
print("ANALYSE DES PERFORMANCES")
print("=" * 50)

# Nombre de paramètres
print(f"Paramètres du modèle Dense: {model_dense.count_params():,}")
print(f"Paramètres du modèle CNN:   {model_cnn.count_params():,}")

# Analyse des erreurs du CNN (si on veut approfondir)
predictions_cnn = model_cnn.predict(x_test_cnn, verbose=0)
pred_labels_cnn = np.argmax(predictions_cnn, axis=1)

# Matrice de confusion simplifiée
errors = np.where(pred_labels_cnn != y_test)[0]
print(f"Nombre d'erreurs du CNN: {len(errors)} / {len(y_test)}")

if len(errors) > 0:
    # Afficher quelques erreurs
    plt.figure(figsize=(12, 4))
    for i, idx in enumerate(errors[:8]):
        plt.subplot(2, 4, i+1)
        plt.imshow(x_test[idx], cmap='gray')
        plt.title(f'Vrai: {y_test[idx]} | Prédit: {pred_labels_cnn[idx]}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()